# HW5 Q2: VGG16 with a customized cost function
Add a regularization term that shrinks the weights of the **first conv layer only** (`model.features[0]`, Conv2d(3, 64, 3×3), 1728 weights):

$$loss = loss_1 + \gamma \times loss_2, \qquad loss_1 = \text{CrossEntropyLoss}, \qquad loss_2 = \sum |W_{conv1}|$$

Every case starts from the same trained HW3 VGG16 checkpoint (step 1) and is fine-tuned with exactly the same settings, so the only difference between cases is $\gamma$.

**For the submission:** after training, do *Kernel → Restart & Clear Output*, then run every cell **except** the training cell.

## Setup (same as HW3)

In [ ]:
import argparse
import os
import time
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

    

import torchvision
import torchvision.transforms as transforms

import sys
sys.path.insert(0, os.path.abspath('..'))   # this notebook is in software/hw/, the models/ folder is in software/
from models import *   # bring everything in the folder models

global best_prec
use_gpu = torch.cuda.is_available()
device = 'cuda' if use_gpu else 'cpu'
print('device:', device)
print('=> Building model...')
    
    
batch_size = 128

model_name = "vgg16"
model = VGG16()

        

normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])


train_dataset = torchvision.datasets.CIFAR10(
    root='../data',
    train=True,
    download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ]))
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)


test_dataset = torchvision.datasets.CIFAR10(
    root='../data',
    train=False,
    download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ]))

testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


print_freq = 100 # every 100 batches, accuracy printed. Here, each batch includes "batch_size" data points
# CIFAR10 has 50,000 training data, and 10,000 validation data.

def train(trainloader, model, criterion, optimizer, epoch):
    batch_time = AverageMeter()   ## at the begining of each epoch, this should be reset
    data_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    model.train()

    end = time.time()  # measure current time
    
    for i, (input, target) in enumerate(trainloader):
        # measure data loading time
        data_time.update(time.time() - end)  # data loading time

        input, target = input.to(device), target.to(device)

        # compute output
        output = model(input)
        loss = criterion(output, target)

        # measure accuracy and record loss
        prec = accuracy(output, target)[0]
        losses.update(loss.item(), input.size(0))
        top1.update(prec.item(), input.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end) # time spent to process one batch
        end = time.time()


        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   epoch, i, len(trainloader), batch_time=batch_time,
                   data_time=data_time, loss=losses, top1=top1))

            

def validate(val_loader, model, criterion ):
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    # switch to evaluate mode
    model.eval()

    end = time.time()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):
         
            input, target = input.to(device), target.to(device)

            # compute output
            output = model(input)
            loss = criterion(output, target)

            # measure accuracy and record loss
            prec = accuracy(output, target)[0]
            losses.update(loss.item(), input.size(0))
            top1.update(prec.item(), input.size(0))

            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if i % print_freq == 0:  # This line shows how frequently print out the status. e.g., i%5 => every 5 batch, prints out
                print('Test: [{0}/{1}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   i, len(val_loader), batch_time=batch_time, loss=losses,
                   top1=top1))

    print(' * Prec {top1.avg:.3f}% '.format(top1=top1))
    return top1.avg


def accuracy(output, target, topk=(1,5)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk) # 5
    batch_size = target.size(0) # 128

    _, pred = output.topk(maxk, 1, True, True) # topk(k, dim=None, largest=True, sorted=True)
                                    # will output (max value, its index)
    pred = pred.t()               # transpose
    correct = pred.eq(target.view(1, -1).expand_as(pred))   # "-1": calculate automatically

    res = []
    for k in topk: # 1, 5
        correct_k = correct[:k].reshape(-1).float().sum(0)  # view(-1): make a flattened 1D tensor
        res.append(correct_k.mul_(100.0 / batch_size))   # correct: size of [maxk, batch_size]
    return res


class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n    ## n is impact factor
        self.count += n
        self.avg = self.sum / self.count

        
def save_checkpoint(state, is_best, fdir):
    filepath = os.path.join(fdir, 'checkpoint.pth')
    torch.save(state, filepath)
    if is_best:
        shutil.copyfile(filepath, os.path.join(fdir, 'model_best.pth.tar'))


def adjust_learning_rate(optimizer, epoch):
    """For resnet, the lr starts from 0.1, and is divided by 10 at 80 and 120 epochs"""
    adjust_list = [150, 225]
    if epoch in adjust_list:
        for param_group in optimizer.param_groups:
            param_group['lr'] = param_group['lr'] * 0.1        

#model = nn.DataParallel(model).cuda()
#all_params = checkpoint['state_dict']
#model.load_state_dict(all_params, strict=False)
#criterion = nn.CrossEntropyLoss().cuda()
#validate(testloader, model, criterion)

In [ ]:
criterion = nn.CrossEntropyLoss().to(device)
step1_path = 'result/'+str(model_name)+'/model_best.pth.tar'   # trained HW3 VGG16 (no quantization)
result_dir = 'result/vgg16_cost_function'                       # fine-tuned models, one per gamma

def load_step1_model():
    # fresh VGG16 with the trained HW3 weights: the starting point of every case
    model = VGG16()
    model.load_state_dict(torch.load(step1_path, map_location=device)['state_dict'])
    return model.to(device)

def conv1_abs_sum(model):
    return model.features[0].weight.abs().sum().item()      # sum of |W| of the first conv layer only

print(load_step1_model().features[0])

## Step 1: initial version (loss = loss1)

In [ ]:
model = load_step1_model()
acc_step1 = validate(testloader, model, criterion)
abs_step1 = conv1_abs_sum(model)
print('Step 1 (initial version): test accuracy = {:.2f}%, conv1 weight abs sum = {:.2f}'.format(acc_step1, abs_step1))

## Customized cost function

In [ ]:
def cost_function(output, target, model, gamma):
    loss1 = criterion(output, target)                  # CrossEntropyLoss
    loss2 = model.features[0].weight.abs().sum()       # sum |W_conv1|: first conv layer only. Kept as a tensor
                                                       # (no .item(), no torch.no_grad()) so its gradient reaches the weights
    loss = loss1 + gamma * loss2
    return loss, loss1, loss2

## Fine-tuning for every γ (run once, then Restart & Clear Output; skip this cell for the submission)
Same settings for every case: start from the step 1 checkpoint, SGD with lr 0.01 and momentum 0.9, constant learning rate, 5 epochs, batch size 128, no weight decay (so the only force shrinking the conv1 weights is $loss_2$), and the same random seed. The model after the last epoch is saved for each γ.

In [ ]:
gammas = [1, 1e-1, 1e-2, 1e-3, 1e-4]   # step 2 (gamma = 1) and step 3 (4 more values, spread over 4 orders of magnitude)
lr = 0.01
epochs = 5
skip_done = True                        # rerunning this cell skips the gammas that already finished

os.makedirs(result_dir, exist_ok=True)

for gamma in gammas:
    path = os.path.join(result_dir, 'gamma_{:g}.pth.tar'.format(gamma))
    if skip_done and os.path.exists(path):
        print('gamma = {:g}: already done, skipping'.format(gamma))
        continue

    torch.manual_seed(0)                # same data order for every case
    model = load_step1_model()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=0)

    print('===== gamma = {:g}'.format(gamma))
    for epoch in range(epochs):
        model.train()
        sum_loss1, sum_loss2, n = 0.0, 0.0, 0
        for input, target in trainloader:
            input, target = input.to(device), target.to(device)
            output = model(input)
            loss, loss1, loss2 = cost_function(output, target, model, gamma)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            sum_loss1 += loss1.item() * input.size(0)
            sum_loss2 += loss2.item() * input.size(0)
            n += input.size(0)
        acc = validate(testloader, model, criterion)
        print('epoch {}: loss1 {:.4f}, loss2 {:.2f}, test acc {:.2f}%, conv1 abs sum {:.2f}'.format(
            epoch, sum_loss1 / n, sum_loss2 / n, acc, conv1_abs_sum(model)))

    torch.save({'gamma': gamma, 'state_dict': model.state_dict()}, path)

## Results: reload every fine-tuned model and evaluate it

In [ ]:
results = {0: (acc_step1, abs_step1)}   # gamma = 0 is the step 1 model
for gamma in [1, 1e-1, 1e-2, 1e-3, 1e-4]:
    model = load_step1_model()
    model.load_state_dict(torch.load(os.path.join(result_dir, 'gamma_{:g}.pth.tar'.format(gamma)),
                                     map_location=device)['state_dict'])
    results[gamma] = (validate(testloader, model, criterion), conv1_abs_sum(model))

print('\n{:>8} {:>15} {:>18}'.format('gamma', 'test acc (%)', 'conv1 abs sum'))
for gamma, (acc, s) in sorted(results.items(), reverse=True):
    print('{:>8g} {:>15.2f} {:>18.2f}'.format(gamma, acc, s))

## Step 3: optimal γ (accuracy above 80%, smallest conv1 weight abs sum)

In [ ]:
candidates = {g: r for g, r in results.items() if g > 0 and r[0] > 80}
gamma_opt = min(candidates, key=lambda g: candidates[g][1])
acc_opt, abs_opt = results[gamma_opt]
print('optimal gamma = {:g}: test accuracy = {:.2f}%, conv1 weight abs sum = {:.2f}'.format(gamma_opt, acc_opt, abs_opt))
print('cost function with the optimal gamma:  loss = CrossEntropyLoss + {:g} x sum|W_conv1|'.format(gamma_opt))

## Summary

In [ ]:
rows = [('1. Initial version',                'loss1',                                      acc_step1,     abs_step1),
        ('2. New loss term (gamma = 1)',      'loss1 + loss2',                              results[1][0], results[1][1]),
        ('3. Fine-tuned (optimal gamma)',     'loss1 + gamma x loss2 (gamma = {:g})'.format(gamma_opt), acc_opt, abs_opt)]
print('{:<32} {:<40} {:>18} {:>22}'.format('Case', 'Cost function', 'Test accuracy (%)', 'Conv1 weight abs sum'))
for case, cost, acc, s in rows:
    print('{:<32} {:<40} {:>18.2f} {:>22.2f}'.format(case, cost, acc, s))